# LeetCode 1496 — Path Crossing bằng Python

**Link đề:** https://leetcode.com/problems/path-crossing/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String, Simulation (mô phỏng)

```python
class Solution:
    def isPathCrossing(self, path: str) -> bool:
        ...
```

### Mục tiêu học
1. Mô phỏng chuyển động trên **mặt phẳng tọa độ** `(x, y)` từ chuỗi lệnh `'N'`, `'S'`, `'E'`, `'W'`.
2. Hiểu **tuple** (bộ giá trị bất biến) và vì sao tuple dùng được làm phần tử của `set`, còn list thì không.
3. Thấy vì sao lưu điểm đã đi vào **list** thì chậm (`O(n²)`), còn **set** thì nhanh (`O(n)` trung bình).
4. Biết dùng `dict` ánh xạ hướng → độ dời `(dx, dy)` để code gọn, và cách dùng **số phức** biểu diễn điểm.
5. Trace, chứng minh, test đối chiếu, tự giải biến thể (đếm điểm trùng, trả về bước đầu tiên bị trùng…).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho chuỗi `path`, mỗi ký tự `path[i]` là `'N'`, `'S'`, `'E'` hoặc `'W'`, nghĩa là đi **một đơn vị** về hướng Bắc, Nam, Đông, Tây.

Bạn bắt đầu tại gốc tọa độ `(0, 0)` và đi theo `path`.

Trả về `True` nếu đường đi **tự cắt** chính nó, tức là có lúc bạn đứng ở một điểm **đã từng đi qua** trước đó. Ngược lại trả về `False`.

| `path` | Các điểm đi qua | Output |
|---|---|---|
| `"NES"` | (0,0) → (0,1) → (1,1) → (1,0) | `False` |
| `"NESWW"` | (0,0) → (0,1) → (1,1) → (1,0) → **(0,0)** → (-1,0) | `True` — quay lại gốc |

**Constraints:** `1 <= path.length <= 10^4`, `path[i]` là một trong `'N'`, `'S'`, `'E'`, `'W'`.

Lưu ý quan trọng: **điểm xuất phát `(0, 0)` cũng tính là đã đi qua.** Đi `"NS"` là quay về `(0,0)` → `True`.

Quy ước trục: `N` tăng `y`, `S` giảm `y`, `E` tăng `x`, `W` giảm `x`.

```
        N (y+1)
          ↑
W (x-1) ← · → E (x+1)
          ↓
        S (y-1)
```

In [1]:
# Đi tay từng bước "NESWW"
x, y = 0, 0
print("bắt đầu:", (x, y))
for c in "NESWW":
    if c == 'N':
        y += 1
    elif c == 'S':
        y -= 1
    elif c == 'E':
        x += 1
    else:            # 'W'
        x -= 1
    print(f"đi {c} -> {(x, y)}")

bắt đầu: (0, 0)
đi N -> (0, 1)
đi E -> (1, 1)
đi S -> (1, 0)
đi W -> (0, 0)
đi W -> (-1, 0)


Nhìn output: sau bước thứ 4 (`W`) ta về lại `(0, 0)` — chính là điểm xuất phát. Vậy đáp án `True`.

## 2. Nền tảng Python cần biết

### 2.1. Tuple — cặp tọa độ bất biến

Tuple viết bằng ngoặc tròn: `p = (1, 2)`. Truy cập như list `p[0]`, `p[1]`, có thể **tách** (unpack) `x, y = p`.

Khác list ở chỗ **bất biến** (immutable): không sửa được `p[0] = 5`. Hai tuple bằng nhau khi mọi thành phần bằng nhau.

In [2]:
p = (1, 2)
x, y = p
print(p, p[0], p[1], "| unpack:", x, y)
print((1, 2) == (1, 2), (1, 2) == (2, 1))
try:
    p[0] = 5
except TypeError as e:
    print("TypeError:", e)

(1, 2) 1 2 | unpack: 1 2
True False
TypeError: 'tuple' object does not support item assignment


### 2.2. `set` — tập hợp để kiểm tra "đã thấy chưa" nhanh

`set` lưu các phần tử **không trùng nhau**. Hai thao tác quan trọng:
- `s.add(x)` — thêm.
- `x in s` — kiểm tra có không: trung bình `O(1)` nhờ **băm** (hashing).

Với list, `x in lst` phải dò từng phần tử → `O(n)`.

In [3]:
visited = {(0, 0)}           # set chứa sẵn điểm gốc
visited.add((0, 1))
visited.add((0, 1))          # thêm trùng không có tác dụng
print(visited, "| số phần tử:", len(visited))
print((0, 1) in visited, (5, 5) in visited)
print("set rỗng phải viết set(), vì {} là dict rỗng:", type(set()), type({}))

{(0, 1), (0, 0)} | số phần tử: 2
True False
set rỗng phải viết set(), vì {} là dict rỗng: <class 'set'> <class 'dict'>


### 2.3. Vì sao tuple bỏ vào set được, còn list thì không?

Phần tử của set phải **hashable** (băm được) — giá trị không đổi theo thời gian. Tuple (chứa số) bất biến → hashable. List có thể sửa → không hashable → báo `TypeError`.

In [4]:
s = set()
s.add((1, 2))
print("thêm tuple OK:", s)
try:
    s.add([1, 2])
except TypeError as e:
    print("thêm list lỗi:", e)
print("hash((1, 2)) =", hash((1, 2)))

thêm tuple OK: {(1, 2)}
thêm list lỗi: unhashable type: 'list'
hash((1, 2)) = -3550055125485641917


### 2.4. `dict` ánh xạ hướng → độ dời

Thay chuỗi `if/elif`, ta tra bảng: mỗi hướng ứng với `(dx, dy)`.

In [5]:
MOVES = {'N': (0, 1), 'S': (0, -1), 'E': (1, 0), 'W': (-1, 0)}
x, y = 0, 0
dx, dy = MOVES['E']
x, y = x + dx, y + dy
print("sau E:", (x, y))
print("MOVES['W'] =", MOVES['W'])

sau E: (1, 0)
MOVES['W'] = (-1, 0)


## 3. Cách 1 — Trực giác: lưu các điểm đã đi vào list

Ý tưởng: giữ list `visited` các điểm đã đến. Mỗi bước tính điểm mới, nếu đã có trong list → `True`; nếu chưa thì thêm vào.

In [6]:
def is_path_crossing_list(path: str) -> bool:
    x, y = 0, 0
    visited = [(0, 0)]               # NHỚ: điểm xuất phát đã được đi qua
    for c in path:
        if c == 'N':
            y += 1
        elif c == 'S':
            y -= 1
        elif c == 'E':
            x += 1
        else:
            x -= 1
        if (x, y) in visited:        # dò tuần tự: O(độ dài list)
            return True
        visited.append((x, y))
    return False

print(is_path_crossing_list("NES"))     # False
print(is_path_crossing_list("NESWW"))   # True

False
True


**Đúng**, nhưng `(x, y) in visited` với list phải so sánh lần lượt. Bước thứ `k` tốn tới `k` phép so sánh → tổng `1 + 2 + ... + n ≈ n²/2`. Với `n = 10^4` là khoảng 50 triệu phép so sánh — chậm rõ rệt trong Python.

## 4. Nút thắt — "đã thấy chưa?" phải trả lời nhanh

Ta chỉ hỏi một câu lặp đi lặp lại: *"điểm này có trong tập đã đi chưa?"*. Đây đúng là việc của **set** (hash set): kiểm tra trung bình `O(1)`.

Đo thử với đường đi dài không tự cắt (đi thẳng về Bắc `n` bước — trường hợp xấu nhất vì không bao giờ dừng sớm):

In [7]:
import time

def is_path_crossing_set_basic(path):
    x, y = 0, 0
    visited = {(0, 0)}
    for c in path:
        if c == 'N': y += 1
        elif c == 'S': y -= 1
        elif c == 'E': x += 1
        else: x -= 1
        if (x, y) in visited:
            return True
        visited.add((x, y))
    return False

long_path = "N" * 10**4
for name, f in [("list", is_path_crossing_list), ("set ", is_path_crossing_set_basic)]:
    t = time.perf_counter()
    r = f(long_path)
    print(f"{name}: kết quả {r}, {(time.perf_counter() - t) * 1000:8.2f} ms")

list: kết quả False,   514.56 ms
set : kết quả False,     1.66 ms


(Thời gian cụ thể tùy máy; cái cần thấy là bản list chậm hơn bản set rất nhiều lần.)

## 5. Cách 2 — Set các tuple + bảng hướng (lời giải chính)

Mổ xẻ từng đoạn:

**Đoạn A — trạng thái ban đầu**
```python
x, y = 0, 0
visited = {(0, 0)}
```

**Đoạn B — đi một bước**
```python
dx, dy = MOVES[c]
x, y = x + dx, y + dy
```

**Đoạn C — kiểm tra TRƯỚC, lưu SAU**
```python
if (x, y) in visited:
    return True
visited.add((x, y))
```
Nếu lưu trước rồi mới kiểm tra, điểm vừa thêm luôn "có trong set" → luôn trả `True` sai.

**Đoạn D — đi hết mà không trùng** → `return False`.

In [8]:
MOVES = {'N': (0, 1), 'S': (0, -1), 'E': (1, 0), 'W': (-1, 0)}

def is_path_crossing_set(path: str) -> bool:
    x, y = 0, 0
    visited = {(0, 0)}
    for c in path:
        dx, dy = MOVES[c]
        x, y = x + dx, y + dy
        if (x, y) in visited:
            return True
        visited.add((x, y))
    return False

print(is_path_crossing_set("NES"))     # False
print(is_path_crossing_set("NESWW"))   # True
print(is_path_crossing_set("NS"))      # True: quay về gốc

False
True
True


## 6. TRACE từng bước

Ví dụ `path = "NESWW"`:

| bước | lệnh | (dx, dy) | điểm mới | có trong visited? | visited sau |
|---:|:-:|---|---|:-:|---|
| 0 | N | (0, 1) | (0, 1) | không | {(0,0), (0,1)} |
| 1 | E | (1, 0) | (1, 1) | không | {…, (1,1)} |
| 2 | S | (0, -1) | (1, 0) | không | {…, (1,0)} |
| 3 | W | (-1, 0) | (0, 0) | **có** → `True` | (dừng) |

Lệnh `W` cuối cùng không bao giờ được xử lý vì đã trả về ở bước 3.

In [9]:
def trace_path_crossing(path):
    x, y = 0, 0
    visited = {(0, 0)}
    print(f"bắt đầu tại {(x, y)}, visited = {visited}")
    for i, c in enumerate(path):
        dx, dy = MOVES[c]
        x, y = x + dx, y + dy
        seen = (x, y) in visited
        print(f"bước {i}: '{c}' ({dx:+d},{dy:+d}) -> {(x, y)} | đã đi qua? {seen}")
        if seen:
            print("=> đường đi tự cắt: True")
            return True
        visited.add((x, y))
    print(f"=> không cắt; đã đi {len(visited)} điểm phân biệt: False")
    return False

trace_path_crossing("NESWW")
print()
trace_path_crossing("NES")

bắt đầu tại (0, 0), visited = {(0, 0)}
bước 0: 'N' (+0,+1) -> (0, 1) | đã đi qua? False
bước 1: 'E' (+1,+0) -> (1, 1) | đã đi qua? False
bước 2: 'S' (+0,-1) -> (1, 0) | đã đi qua? False
bước 3: 'W' (-1,+0) -> (0, 0) | đã đi qua? True
=> đường đi tự cắt: True

bắt đầu tại (0, 0), visited = {(0, 0)}
bước 0: 'N' (+0,+1) -> (0, 1) | đã đi qua? False
bước 1: 'E' (+1,+0) -> (1, 1) | đã đi qua? False
bước 2: 'S' (+0,-1) -> (1, 0) | đã đi qua? False
=> không cắt; đã đi 4 điểm phân biệt: False


False

`{dx:+d}` trong f-string in số **kèm dấu** (`+1`, `-1`, `+0`) để dễ đọc độ dời.

Trace một đường cắt **không phải tại gốc** — đi vòng rồi chạm lại điểm giữa:

In [10]:
trace_path_crossing("NNESWS")   # (0,1) bị đi lại ở bước 4

bắt đầu tại (0, 0), visited = {(0, 0)}
bước 0: 'N' (+0,+1) -> (0, 1) | đã đi qua? False
bước 1: 'N' (+0,+1) -> (0, 2) | đã đi qua? False
bước 2: 'E' (+1,+0) -> (1, 2) | đã đi qua? False
bước 3: 'S' (+0,-1) -> (1, 1) | đã đi qua? False
bước 4: 'W' (-1,+0) -> (0, 1) | đã đi qua? True
=> đường đi tự cắt: True


True

## 7. Cách 3 — Biểu diễn điểm bằng số phức

Python có sẵn **số phức** (complex): `a + bj` với `j` là đơn vị ảo. Ta coi phần thực là `x`, phần ảo là `y`. Đi một bước chỉ là **cộng một số phức**:

- `N` → `+1j`, `S` → `-1j`, `E` → `+1`, `W` → `-1`.

Số phức cũng hashable nên bỏ vào set được. Cách này gọn, hay gặp trong các lời giải ngắn.

In [11]:
p = 0 + 0j
print(p, type(p))
p += 1j          # N
p += 1           # E
print("sau NE:", p, "| x =", p.real, "| y =", p.imag)

0j <class 'complex'>
sau NE: (1+1j) | x = 1.0 | y = 1.0


In [12]:
STEP = {'N': 1j, 'S': -1j, 'E': 1, 'W': -1}

def is_path_crossing_complex(path: str) -> bool:
    pos = 0j
    visited = {pos}
    for c in path:
        pos += STEP[c]
        if pos in visited:
            return True
        visited.add(pos)
    return False

print(is_path_crossing_complex("NES"), is_path_crossing_complex("NESWW"))

False True


Vì tọa độ là số nguyên nhỏ (|x|, |y| ≤ 10^4), phần thực/ảo dạng float vẫn **chính xác tuyệt đối**, không lo sai số.

## 8. Cách 4 — So sánh số điểm phân biệt (không dừng sớm)

Một góc nhìn khác: đường đi có `n + 1` "lần đứng" (tính cả điểm gốc). Đường **không** tự cắt ⇔ tất cả `n + 1` điểm đều **phân biệt** ⇔ `len(set(các điểm)) == n + 1`.

Cách này luôn đi hết chuỗi (không dừng sớm), nhưng là bài học tốt về "đếm phần tử phân biệt bằng set".

In [13]:
def is_path_crossing_count(path: str) -> bool:
    x = y = 0
    points = [(0, 0)]
    for c in path:
        dx, dy = MOVES[c]
        x, y = x + dx, y + dy
        points.append((x, y))
    return len(set(points)) < len(points)   # có trùng ⇔ số phân biệt ít hơn tổng

print(is_path_crossing_count("NES"), is_path_crossing_count("NESWW"))

False True


## 9. Vì sao đúng?

**Invariant** của cách 2: ngay trước khi xử lý ký tự thứ `i`, `visited` chứa **đúng** tập các điểm đã đứng tại thời điểm `0, 1, ..., i` (gồm gốc), và các điểm này **đôi một khác nhau** (nếu không ta đã `return True` trước đó).

- Bước `i` tính đúng điểm mới `(x, y)` (cộng độ dời của hướng).
- Nếu `(x, y) ∈ visited` → điểm mới trùng một điểm đã đứng trước → đúng định nghĩa "tự cắt" → `True`.
- Nếu không → thêm vào, invariant giữ nguyên cho bước sau.
- Đi hết chuỗi mà không gặp trùng → mọi điểm đều phân biệt → `False`.

Không bỏ sót: mỗi điểm mới đều được so với **toàn bộ** lịch sử (nhờ set). Không báo nhầm: chỉ trả `True` khi thật sự có trùng.

## 10. Code nộp LeetCode

In [14]:
class Solution:
    def isPathCrossing(self, path: str) -> bool:
        moves = {'N': (0, 1), 'S': (0, -1), 'E': (1, 0), 'W': (-1, 0)}
        x = y = 0
        visited = {(0, 0)}
        for c in path:
            dx, dy = moves[c]
            x += dx
            y += dy
            if (x, y) in visited:
                return True
            visited.add((x, y))
        return False

sol = Solution()
assert sol.isPathCrossing("NES") is False
assert sol.isPathCrossing("NESWW") is True
print("PASS ví dụ đề")

PASS ví dụ đề


`x = y = 0` là **gán dây chuyền**: cả hai biến cùng nhận giá trị 0 (an toàn với số vì số bất biến).

## 11. Phân tích Time / Space

Gọi `n = len(path)`.

| Cách | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. List visited | `O(n²)` worst | `O(n)` | `in` trên list là dò tuần tự |
| 2. Set tuple | `O(n)` trung bình | `O(n)` | lời giải chuẩn |
| 3. Set số phức | `O(n)` trung bình | `O(n)` | gọn |
| 4. Đếm phân biệt | `O(n)` trung bình | `O(n)` | luôn đi hết chuỗi |

"Trung bình" vì thao tác set dựa vào băm: thường `O(1)`, trường hợp va chạm băm cực xấu (hiếm) có thể chậm hơn. Bộ nhớ `O(n)` vì trong trường hợp xấu nhất (không cắt) phải lưu `n + 1` điểm.

## 12. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<34} -> {actual}")

tests = [
    ("đề 1",                           "NES",      False),
    ("đề 2 (về gốc)",                  "NESWW",    True),
    ("1 bước",                         "N",        False),
    ("đi rồi quay lại ngay",           "NS",       True),
    ("EW",                             "EW",       True),
    ("đi thẳng dài",                   "E" * 50,   False),
    ("hình vuông kín",                 "NESW",     True),
    ("cắt điểm giữa, không phải gốc",  "NNESWS",   True),
    ("xoắn ốc không cắt",              "NESSWWNNN", False),
]
for name, p, exp in tests:
    for f in (is_path_crossing_list, is_path_crossing_set, is_path_crossing_complex, is_path_crossing_count):
        assert f(p) == exp, (f.__name__, p)
    check(name, Solution().isPathCrossing(p), exp)

PASS  đề 1                               -> False
PASS  đề 2 (về gốc)                      -> True
PASS  1 bước                             -> False
PASS  đi rồi quay lại ngay               -> True
PASS  EW                                 -> True
PASS  đi thẳng dài                       -> False
PASS  hình vuông kín                     -> True
PASS  cắt điểm giữa, không phải gốc      -> True
PASS  xoắn ốc không cắt                  -> False


In [16]:
# Đối chiếu ngẫu nhiên 4 cách
import random
random.seed(2026)
for _ in range(3000):
    p = "".join(random.choice("NSEW") for _ in range(random.randint(1, 25)))
    exp = is_path_crossing_list(p)
    assert is_path_crossing_set(p) == exp
    assert is_path_crossing_complex(p) == exp
    assert is_path_crossing_count(p) == exp
    assert Solution().isPathCrossing(p) == exp
print("3000 đường đi ngẫu nhiên: 4 cách khớp nhau. PASS")

3000 đường đi ngẫu nhiên: 4 cách khớp nhau. PASS


`random.choice("NSEW")` chọn ngẫu nhiên một ký tự; `"".join(...)` ghép các ký tự thành chuỗi.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên thêm điểm gốc `(0, 0)` vào visited
```python
visited = set()     # SAI
```
Khi đó `"NS"` (quay về gốc) bị trả `False`.

### Lỗi 2: Thêm trước, kiểm tra sau
```python
visited.add((x, y))
if (x, y) in visited:   # SAI: luôn đúng
    return True
```

### Lỗi 3: Dùng list `[x, y]` làm phần tử set → `TypeError: unhashable type: 'list'`.

### Lỗi 4: Chỉ kiểm tra "về lại gốc"
```python
return x == 0 and y == 0   # SAI: đây là bài 657 Robot Return to Origin, khác bài này
```
Đường `"NNESWS"` cắt tại `(0,1)` nhưng không kết thúc ở gốc; còn `"NESWW"` ở bước cuối không đứng ở gốc nhưng đã đi qua gốc.

### Lỗi 5: Đặt trục tùy tiện (ví dụ cho `N` tăng `x`)
Đáp án vẫn đúng nếu **nhất quán** (đổi trục đồng loạt không làm thay đổi việc hai điểm có trùng nhau hay không), nhưng trace sẽ khó đọc và dễ nhầm khi debug. Nên theo quy ước toán học: `N` là `y + 1`, `E` là `x + 1`.

In [17]:
def wrong_no_origin(path):
    x = y = 0
    visited = set()                       # lỗi 1
    for c in path:
        dx, dy = MOVES[c]; x += dx; y += dy
        if (x, y) in visited:
            return True
        visited.add((x, y))
    return False

def wrong_add_first(path):
    x = y = 0
    visited = {(0, 0)}
    for c in path:
        dx, dy = MOVES[c]; x += dx; y += dy
        visited.add((x, y))               # lỗi 2
        if (x, y) in visited:
            return True
    return False

def wrong_only_origin(path):
    x = y = 0
    for c in path:
        dx, dy = MOVES[c]; x += dx; y += dy
    return x == 0 and y == 0              # lỗi 4

print("lỗi 1 'NS'    :", wrong_no_origin("NS"), "| đúng:", is_path_crossing_set("NS"))
print("lỗi 2 'NES'   :", wrong_add_first("NES"), "| đúng:", is_path_crossing_set("NES"))
print("lỗi 4 'NESWW' :", wrong_only_origin("NESWW"), "| đúng:", is_path_crossing_set("NESWW"))
print("lỗi 4 'NNESWS':", wrong_only_origin("NNESWS"), "| đúng:", is_path_crossing_set("NNESWS"))
try:
    {[0, 0]}
except TypeError as e:
    print("lỗi 3:", e)

lỗi 1 'NS'    : False | đúng: True
lỗi 2 'NES'   : True | đúng: False
lỗi 4 'NESWW' : False | đúng: True
lỗi 4 'NNESWS': True | đúng: True
lỗi 3: unhashable type: 'list'


## 14. Mẹo nhận dạng pattern

- "Đã từng ở trạng thái này chưa?" / "có lặp lại không?" → **set các trạng thái đã thấy** (visited set).
- Trạng thái gồm nhiều thành phần (tọa độ, (vị trí, hướng), …) → đóng gói thành **tuple** làm khóa.
- Di chuyển theo hướng trên lưới → **bảng độ dời** `{hướng: (dx, dy)}` thay vì chuỗi `if`.
- Lưới 2D đơn giản → có thể dùng **số phức** cho gọn.

Các bài cùng họ: 657 Robot Return to Origin, 874 Walking Robot Simulation, 202 Happy Number (visited set cho chu trình), 1041 Robot Bounded In Circle.

## 15. Biến thể

### Biến thể A — Trả về chỉ số bước đầu tiên bị trùng (hoặc -1)

In [18]:
def first_crossing_step(path):
    x = y = 0
    visited = {(0, 0)}
    for i, c in enumerate(path):
        dx, dy = MOVES[c]; x += dx; y += dy
        if (x, y) in visited:
            return i
        visited.add((x, y))
    return -1

print(first_crossing_step("NESWW"), first_crossing_step("NES"))   # 3 -1

3 -1


### Biến thể B — Robot Return to Origin (LeetCode 657)
Chỉ hỏi **kết thúc** có ở gốc không: không cần set, chỉ cần đếm `N == S` và `E == W`.

In [19]:
def judge_circle(moves):
    return moves.count('U') == moves.count('D') and moves.count('L') == moves.count('R')

print(judge_circle("UD"), judge_circle("LL"))   # True False

True False


## 16. Luyện tập — 6 bài từ dễ đến nâng cao

### Bài 1 (đoán output)
Không chạy, đoán tọa độ cuối của `"NNWSE"`. Trả về tuple.

In [20]:
def exercise_final_position() -> tuple[int, int]:
    # TODO
    pass

# assert exercise_final_position() == (0, 1)

### Bài 2 (điền chỗ trống)

In [21]:
def exercise_is_path_crossing(path: str) -> bool:
    moves = {'N': (0, 1), 'S': (0, -1), 'E': (1, 0), 'W': (-1, 0)}
    x = y = 0
    # TODO 1: visited = ?
    for c in path:
        # TODO 2: cập nhật x, y
        # TODO 3: kiểm tra trước, lưu sau
        pass
    return False

# assert exercise_is_path_crossing("NES") is False
# assert exercise_is_path_crossing("NESWW") is True

### Bài 3 (đếm số điểm phân biệt)
Trả về số điểm **phân biệt** đã đứng (tính cả gốc).

Ví dụ `"NESWW"` → các điểm (0,0),(0,1),(1,1),(1,0),(0,0),(-1,0) → **5** điểm phân biệt.

In [22]:
def exercise_count_distinct(path: str) -> int:
    # TODO
    pass

# assert exercise_count_distinct("NESWW") == 5
# assert exercise_count_distinct("NS") == 2

### Bài 4 (đếm số lần "giẫm lại")
Đếm số bước mà điểm mới **đã từng** được đứng trước đó (không dừng sớm).

Ví dụ `"NSNS"`: (0,1) mới, (0,0) cũ, (0,1) cũ, (0,0) cũ → **3**.

In [23]:
def exercise_count_revisits(path: str) -> int:
    # TODO
    pass

# assert exercise_count_revisits("NSNS") == 3
# assert exercise_count_revisits("NES") == 0

### Bài 5 (bước lớn hơn 1)
Mỗi lệnh có dạng `"<hướng><số bước>"`, cách nhau bởi dấu cách, ví dụ `"N2 E1 S3 W1"`. Mỗi lệnh đi **từng ô một**. Trả về `True` nếu có ô nào bị đi qua hai lần (tính cả gốc).

In [24]:
def exercise_path_crossing_steps(commands: str) -> bool:
    # TODO: split(), lấy hướng = cmd[0], số bước = int(cmd[1:])
    pass

# assert exercise_path_crossing_steps("N2 E1 S3 W1") is False
# assert exercise_path_crossing_steps("N2 E1 S1 W2 E1") is True

### Bài 6 (mini challenge — cắt cạnh, không chỉ đỉnh)
Đề gốc chỉ xét **điểm**. Giờ hãy xét **đoạn đường**: trả về `True` nếu có **đoạn đơn vị** (cạnh nối 2 điểm kề nhau) bị đi lại lần hai, theo bất kỳ chiều nào.

Ví dụ `"NS"` → đoạn (0,0)–(0,1) đi lại → `True`. `"NESW"` → 4 cạnh hình vuông khác nhau → `False` (dù về lại gốc!).

In [25]:
def exercise_edge_crossing(path: str) -> bool:
    # TODO: khóa của một cạnh không phụ thuộc chiều đi
    pass

# assert exercise_edge_crossing("NS") is True
# assert exercise_edge_crossing("NESW") is False
# assert exercise_edge_crossing("NESWN") is True

### Gợi ý ba mức

**Bài 1** — M1: N/S đổi y, E/W đổi x. M2: hai `N` → y=2, `S` → y=1. M3: `W` rồi `E` triệt tiêu → x=0.

**Bài 2** — M1: nhớ điểm gốc. M2: `visited = {(0, 0)}`, `dx, dy = moves[c]`. M3: `if (x, y) in visited: return True` rồi `visited.add((x, y))`.

**Bài 3** — M1: đừng dừng sớm. M2: luôn `add`. M3: `return len(visited)`.

**Bài 4** — M1: kiểm tra trước khi add, cộng đếm nếu đã có. M2: `if (x, y) in visited: cnt += 1`. M3: `visited.add(...)` ở mọi bước (thêm trùng không sao).

**Bài 5** — M1: `commands.split()` cho list lệnh. M2: `d, k = cmd[0], int(cmd[1:])`. M3: lặp `for _ in range(k):` đi một ô, kiểm tra như bài gốc.

**Bài 6** — M1: một cạnh xác định bởi 2 đầu mút. M2: chiều đi không quan trọng → dùng `frozenset({p, q})` hoặc `tuple(sorted((p, q)))` làm khóa. M3: lưu set các cạnh, kiểm tra trước khi thêm.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [26]:
def answer_final_position():
    return (0, 1)

def answer_is_path_crossing(path):
    moves = {'N': (0, 1), 'S': (0, -1), 'E': (1, 0), 'W': (-1, 0)}
    x = y = 0
    visited = {(0, 0)}
    for c in path:
        dx, dy = moves[c]
        x, y = x + dx, y + dy
        if (x, y) in visited:
            return True
        visited.add((x, y))
    return False

def answer_count_distinct(path):
    x = y = 0
    visited = {(0, 0)}
    for c in path:
        dx, dy = MOVES[c]; x += dx; y += dy
        visited.add((x, y))
    return len(visited)

def answer_count_revisits(path):
    x = y = 0
    visited = {(0, 0)}
    cnt = 0
    for c in path:
        dx, dy = MOVES[c]; x += dx; y += dy
        if (x, y) in visited:
            cnt += 1
        visited.add((x, y))
    return cnt

def answer_path_crossing_steps(commands):
    x = y = 0
    visited = {(0, 0)}
    for cmd in commands.split():
        d, k = cmd[0], int(cmd[1:])
        dx, dy = MOVES[d]
        for _ in range(k):
            x, y = x + dx, y + dy
            if (x, y) in visited:
                return True
            visited.add((x, y))
    return False

def answer_edge_crossing(path):
    x = y = 0
    edges = set()
    for c in path:
        dx, dy = MOVES[c]
        nx, ny = x + dx, y + dy
        key = frozenset({(x, y), (nx, ny)})   # tập 2 đầu mút: không phụ thuộc chiều
        if key in edges:
            return True
        edges.add(key)
        x, y = nx, ny
    return False

assert answer_final_position() == (0, 1)
assert answer_is_path_crossing("NES") is False and answer_is_path_crossing("NESWW") is True
assert answer_count_distinct("NESWW") == 5 and answer_count_distinct("NS") == 2
assert answer_count_revisits("NSNS") == 3 and answer_count_revisits("NES") == 0
assert answer_path_crossing_steps("N2 E1 S3 W1") is False
assert answer_path_crossing_steps("N2 E1 S1 W2 E1") is True
assert answer_edge_crossing("NS") is True
assert answer_edge_crossing("NESW") is False
assert answer_edge_crossing("NESWN") is True
for _ in range(500):
    p = "".join(random.choice("NSEW") for _ in range(random.randint(1, 20)))
    assert answer_is_path_crossing(p) == is_path_crossing_set(p)
    assert (answer_count_distinct(p) < len(p) + 1) == is_path_crossing_set(p)
print("frozenset({(0,0),(0,1)}) == frozenset({(0,1),(0,0)}):", frozenset({(0, 0), (0, 1)}) == frozenset({(0, 1), (0, 0)}))
print("Tất cả đáp án PASS!")

frozenset({(0,0),(0,1)}) == frozenset({(0,1),(0,0)}): True
Tất cả đáp án PASS!


`frozenset` là **set bất biến** — hashable nên dùng làm khóa được, và không phân biệt thứ tự phần tử.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isPathCrossing(self, path: str) -> bool:
        moves = {'N': (0, 1), 'S': (0, -1), 'E': (1, 0), 'W': (-1, 0)}
        x = y = 0
        visited = {(0, 0)}               # gốc đã đi qua
        for c in path:
            dx, dy = moves[c]
            x, y = x + dx, y + dy
            if (x, y) in visited:        # kiểm tra TRƯỚC
                return True
            visited.add((x, y))          # lưu SAU
        return False
```

**Checklist trước khi submit:**
- [ ] `visited` có sẵn `(0, 0)`?
- [ ] Kiểm tra trước, thêm sau?
- [ ] Phần tử set là **tuple**, không phải list?
- [ ] Dùng `set`, không dùng list (tránh `O(n²)`)?
- [ ] Phân biệt "đi qua điểm cũ" với "kết thúc ở gốc"?